# Time-domain Maxwell MHM in two and three dimensions

Tangential hybridization of central DG operators, physical material masses and leapfrog evolution. Electric and magnetic fields are compared at their different staggered times. Broken curl errors do not establish global H(curl) conformity.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
if not (ROOT / "pixi.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


In [ ]:
from pymhm.tetrahedral import TetraMesh
from pymhm.maxwell import MaxwellStepper
from examples.maxwell_data import CavityMode

mode = CavityMode(3)
with threadpool_limits(1), MaxwellStepper(TetraMesh.unit_cube(), time_step=0.001) as stepper:
    initial = stepper.initialize(mode.electric_shape)
    for _ in range(5):
        solution = stepper.advance()
        assert abs(solution.energy_balance_residual) < 2e-13
    drift = abs(solution.energy / initial.energy - 1)
assert drift < 2e-13
{"electric_time": solution.electric_time, "magnetic_time": solution.magnetic_time,
 "relative_cross_time_energy_drift": drift}

## Five-level spatial and temporal studies

The TM cavity uses the analytical data of Lanteri et al. (2018), one local P(ell+2) element and polynomial face degree ell. The three-dimensional mode is an original full-vector verification. The temporal baseline is the exact exponential of the same constrained spatial ODE.

In [ ]:
record = json.loads((ROOT / "examples/results/maxwell/comparison.json").read_text())
assert record["source_changed_during_run"] is False
[(r["dimension"], r["resolution"], r["trace_degree"], r["max_in_time"]["combined_l2"],
  r["max_in_time"]["combined_hcurl"], r["modified_energy_relative_drift"])
 for r in record["rows"] if r["study"] == "cavity"]

In [ ]:
for name in ("convergence", "time-energy", "cavity-2d-ell2-n8-fields", "cavity-3d-ell1-n4-electric", "cavity-3d-ell1-n4-magnetic"):
    display(Image(filename=str(ROOT / f"docs/figures/maxwell/{name}.png")))

Every spatial panel preserves signs and incident DG values and overlays the actual macro mesh or its physical section. Energy conservation, algebraic residuals and field accuracy are distinct checks. The quadrilateral heterogeneous nano-waveguide is a separate benchmark.